# Inspecting the Self-Attention Mechanism

A hands-on exploration of the scaled dot-product self-attention used in Transformer models. We'll reimplement the core attention function, hook into a Hugging Face Transformer to log and visualize attention matrices, experiment with alternative scoring functions, and benchmark speed and accuracy trade-offs.

## Why Self-Attention is Central to Transformers

Self-attention is the core innovation that enables Transformers to:
- **Capture long-range dependencies** without recurrence
- **Process sequences in parallel** for faster training
- **Learn complex relationships** between any two positions
- **Enable interpretability** through attention weight visualization

In this notebook, we'll dissect how self-attention works under the hood and explore its variants and optimizations.

## Objective

**The question this notebook answers:** when a Transformer "attends", what is actually being
computed, and how much of the story the attention heatmaps tell can you rely on?

Concretely, we do five things and check each one against a number rather than a picture:

| Step | What we build | How we check it |
|---|---|---|
| 1 | Scaled dot-product and multi-head attention from scratch | Attention rows sum to 1; output shapes match the inputs |
| 2 | Attention extracted from a real pretrained model | Layer/head/sequence dimensions reported straight from the model |
| 3 | Three alternative scoring functions (additive, bilinear, local) | Same interface, compared on sparsity and wall-clock cost |
| 4 | A speed and memory benchmark against PyTorch's fused kernel | Milliseconds per forward pass across three sequence lengths |
| 5 | A head-ablation study on DistilBERT | Cosine drift in the sentence embedding when one head is switched off |

**What success looks like:** by the end you should be able to write the attention kernel from
memory, pull attention out of any Hugging Face model, and say precisely why the scratch
version is slower than the library one and why an attention map is not an explanation.

**What this notebook is not:** it is not a training run. Nothing here is fine-tuned and no
accuracy is reported, because every question above is answerable from forward passes alone.

**Runtime:** CPU is enough. The models used are small (DistilBERT, a RoBERTa sentiment head)
and are downloaded from the Hugging Face Hub on first use, so the notebook needs internet
enabled but no accelerator.

In [ ]:
# Setup and Installation
import sys
import subprocess
import time
import math
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Install required packages
packages = ['torch', 'transformers', 'seaborn']
for package in packages:
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, BertModel, GPT2Model
import seaborn as sns

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("✅ All dependencies installed successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers available: {'✅' if 'transformers' in sys.modules else '❌'}")
print(f"📊 Seaborn available: {'✅' if 'seaborn' in sys.modules else '❌'}")

# Check device availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

## Scaled Dot-Product Attention from Scratch

The core self-attention mechanism computes attention weights using three learned projections:

- **Queries (Q)**: "What am I looking for?"
- **Keys (K)**: "What do I represent?"  
- **Values (V)**: "What information do I contain?"

**Mathematical Formula:**
Attention(Q, K, V) = softmax(QK^T / √d_k)V


**Key components:**
- **Scaling factor (√d_k)**: Prevents softmax saturation for large dimensions
- **Softmax**: Ensures attention weights sum to 1
- **Masking**: Prevents attention to future tokens (causal attention) or padding

In [ ]:
def scaled_dot_product_attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, 
                               mask: torch.Tensor = None, 
                               dropout: float = 0.0) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Implement scaled dot-product attention from scratch.
    
    Args:
        Q: Query tensor [batch_size, seq_len, d_k]
        K: Key tensor [batch_size, seq_len, d_k]  
        V: Value tensor [batch_size, seq_len, d_v]
        mask: Optional mask tensor [batch_size, seq_len, seq_len]
        dropout: Dropout probability for attention weights
        
    Returns:
        output: Attention output [batch_size, seq_len, d_v]
        attention_weights: Attention probabilities [batch_size, seq_len, seq_len]
    """
    # Calculate attention scores
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    
    # Apply mask if provided
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Apply softmax to get attention weights
    attention_weights = torch.softmax(scores, dim=-1)
    
    # Apply dropout if specified
    if dropout > 0.0:
        attention_weights = F.dropout(attention_weights, p=dropout, training=True)
    
    # Apply attention weights to values
    output = attention_weights @ V
    
    return output, attention_weights

def multi_head_attention(x: torch.Tensor, num_heads: int, d_model: int, 
                        mask: torch.Tensor = None) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Multi-head attention implementation.
    
    Args:
        x: Input tensor [batch_size, seq_len, d_model]
        num_heads: Number of attention heads
        d_model: Model dimension
        mask: Optional attention mask
        
    Returns:
        output: Multi-head attention output
        attention_weights: Averaged attention weights across heads
    """
    batch_size, seq_len, _ = x.shape
    d_k = d_model // num_heads
    
    # Linear projections for Q, K, V
    W_q = nn.Linear(d_model, d_model, bias=False)
    W_k = nn.Linear(d_model, d_model, bias=False)
    W_v = nn.Linear(d_model, d_model, bias=False)
    W_o = nn.Linear(d_model, d_model, bias=False)
    
    # Project to Q, K, V
    Q = W_q(x).view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    K = W_k(x).view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    V = W_v(x).view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    
    # Apply attention
    attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
    
    # Concatenate heads
    attn_output = attn_output.transpose(1, 2).contiguous().view(
        batch_size, seq_len, d_model)
    
    # Final linear projection
    output = W_o(attn_output)
    
    # Average attention weights across heads for visualization
    avg_attn_weights = attn_weights.mean(dim=1)
    
    return output, avg_attn_weights

# Test with random tensors
print("🧪 Testing scaled dot-product attention...")

batch_size, seq_len, d_model = 2, 8, 64
num_heads = 8

# Create random input
x = torch.randn(batch_size, seq_len, d_model)
print(f"📊 Input shape: {x.shape}")

# Test single-head attention
Q = torch.randn(batch_size, seq_len, d_model)
K = torch.randn(batch_size, seq_len, d_model)
V = torch.randn(batch_size, seq_len, d_model)

output, weights = scaled_dot_product_attention(Q, K, V)
print(f"📈 Single-head output shape: {output.shape}")
print(f"📈 Attention weights shape: {weights.shape}")
print(f"📊 Attention weights statistics:")
print(f"  - Mean: {weights.mean():.4f}")
print(f"  - Std: {weights.std():.4f}")
print(f"  - Row sums (should be ~1.0): {weights.sum(dim=-1).mean():.4f}")

# Test multi-head attention
mh_output, mh_weights = multi_head_attention(x, num_heads, d_model)
print(f"\n🔀 Multi-head output shape: {mh_output.shape}")
print(f"🔀 Multi-head weights shape: {mh_weights.shape}")

print("\n✅ Attention implementation tests passed!")

## Data Overview

There is no dataset to download here; the notebook runs on three kinds of input, and it is
worth being clear about which section uses which, because the conclusions you can draw
differ sharply between them.

**1. Random tensors** (`torch.randn`), used for the from-scratch implementation, the
mechanism comparison and the benchmark. Seeded with `torch.manual_seed(42)` in the setup
cell. Random input is the right choice for those sections because they are testing shapes,
normalisation and speed -- properties that do not depend on the data being meaningful. It is
the wrong choice for anything about linguistics, so no claim about language is made from
these cells.

**2. Short English sentences**, used wherever real attention patterns are inspected:
- `"The cat sat on the mat and looked around."` for the DistilBERT visualisation, chosen
  because it has repeated determiners and a clear subject-verb structure, which is what
  makes head specialisation visible in a heatmap.
- Four sentiment-bearing sentences (strongly positive, strongly negative, hedged, and
  emphatic) for the case study, chosen so the hedged one gives the model something to be
  uncertain about.
- Four short factual sentences for the ablation study, kept short so every head sees a
  sequence with no padding.

**3. Pretrained model weights**, which are the real "data" in the sense that they are what
the attention patterns are a property of:
- `distilbert-base-uncased` -- 6 layers, 12 heads per layer; used for visualisation and
  ablation. Small enough to ablate exhaustively on CPU.
- `cardiffnlp/twitter-roberta-base-sentiment-latest` -- a 3-class sentiment head; used for
  the case study.

Sequence lengths are short throughout (roughly 8-16 tokens after WordPiece), which keeps the
heatmaps legible. The benchmark section deliberately breaks that rule and sweeps to 256
tokens, because the quadratic memory cost only becomes visible at length.

## Hooking into Hugging Face's Attention

Hugging Face Transformers provide pre-trained models with optimized attention implementations. We'll hook into the attention mechanism to capture and analyze attention patterns from real models.

**Key steps:**
1. Load a pre-trained model with `output_attentions=True`
2. Create custom hooks to capture attention weights
3. Process sample text and extract attention matrices
4. Analyze attention patterns across layers and heads

In [ ]:
class AttentionHook:
    """Hook to capture attention weights from Hugging Face models."""
    
    def __init__(self):
        self.attention_weights = []
        self.layer_names = []
    
    def clear(self):
        """Clear stored attention weights."""
        self.attention_weights = []
        self.layer_names = []
    
    def hook_fn(self, module, input, output):
        """Hook function to capture attention weights."""
        if hasattr(output, 'attentions') and output.attentions is not None:
            self.attention_weights.append(output.attentions.detach())
            self.layer_names.append(module.__class__.__name__)

def load_model_with_hooks(model_name: str = "bert-base-uncased"):
    """Load model and tokenizer with attention hooks."""
    print(f"🤗 Loading {model_name}...")
    
    # Load model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name, output_attentions=True)
    model.eval()
    
    # Create attention hook
    attention_hook = AttentionHook()
    
    # Register hooks on attention layers
    hooks = []
    for name, module in model.named_modules():
        if 'attention' in name.lower() and hasattr(module, 'self'):
            hook = module.register_forward_hook(attention_hook.hook_fn)
            hooks.append(hook)
    
    print(f"✅ Registered {len(hooks)} attention hooks")
    return model, tokenizer, attention_hook, hooks

def analyze_attention_on_text(model, tokenizer, attention_hook, text: str):
    """Analyze attention patterns on input text."""
    print(f"🔍 Analyzing attention for: '{text}'")
    
    # Clear previous hooks
    attention_hook.clear()
    
    # Tokenize input
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    print(f"🎯 Tokens ({len(tokens)}): {tokens}")
    
    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract attention weights
    attentions = outputs.attentions  # List of attention tensors per layer
    
    print(f"📊 Captured attention from {len(attentions)} layers")
    print(f"📏 Attention shape per layer: {attentions[0].shape}")  # [batch, heads, seq, seq]
    
    return attentions, tokens, inputs

# Load model and set up hooks
model, tokenizer, attention_hook, hooks = load_model_with_hooks("distilbert-base-uncased")

# Test with sample text
sample_text = "The cat sat on the mat and looked around."
attentions, tokens, inputs = analyze_attention_on_text(model, tokenizer, attention_hook, sample_text)

print(f"\n📋 Model Information:")
print(f"  - Model type: {model.__class__.__name__}")
print(f"  - Number of layers: {len(attentions)}")
print(f"  - Number of heads: {attentions[0].shape[1]}")
print(f"  - Sequence length: {attentions[0].shape[2]}")

## Visualizing Attention Maps

Attention visualization helps us understand:
- **Which tokens** the model focuses on for each prediction
- **Head specialization** - different heads learning different patterns
- **Layer differences** - how attention evolves through the network
- **Linguistic patterns** - syntactic vs. semantic relationships

In [ ]:
def plot_attention_head(attention_weights: torch.Tensor, tokens: list[str], 
                       layer_idx: int, head_idx: int, figsize: tuple = (10, 8)):
    """Plot attention weights for a specific layer and head."""
    
    # Extract attention for specific layer and head
    attn = attention_weights[layer_idx][0, head_idx].numpy()  # [seq_len, seq_len]
    
    # Create heatmap
    plt.figure(figsize=figsize)
    sns.heatmap(attn, 
                xticklabels=tokens, 
                yticklabels=tokens,
                cmap='Blues', 
                annot=True if len(tokens) <= 10 else False,
                fmt='.2f',
                cbar_kws={'label': 'Attention Weight'})
    
    plt.title(f'Attention Weights - Layer {layer_idx}, Head {head_idx}', 
              fontsize=14, fontweight='bold')
    plt.xlabel('Key Tokens', fontweight='bold')
    plt.ylabel('Query Tokens', fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

def plot_attention_overview(attention_weights: list[torch.Tensor], tokens: list[str], 
                          max_layers: int = 6, max_heads: int = 4):
    """Plot overview of attention patterns across layers and heads."""
    
    num_layers = min(len(attention_weights), max_layers)
    num_heads = min(attention_weights[0].shape[1], max_heads)
    
    fig, axes = plt.subplots(num_layers, num_heads, 
                            figsize=(4 * num_heads, 3 * num_layers))
    
    if num_layers == 1:
        axes = axes.reshape(1, -1)
    if num_heads == 1:
        axes = axes.reshape(-1, 1)
    
    for layer in range(num_layers):
        for head in range(num_heads):
            ax = axes[layer, head]
            attn = attention_weights[layer][0, head].numpy()
            
            im = ax.imshow(attn, cmap='Blues', aspect='auto')
            ax.set_title(f'L{layer} H{head}', fontsize=10)
            
            if layer == num_layers - 1:  # Bottom row
                ax.set_xticks(range(len(tokens)))
                ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
            else:
                ax.set_xticks([])
            
            if head == 0:  # Leftmost column
                ax.set_yticks(range(len(tokens)))
                ax.set_yticklabels(tokens, fontsize=8)
            else:
                ax.set_yticks([])
    
    plt.suptitle('Attention Patterns Across Layers and Heads', 
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

def analyze_attention_statistics(attention_weights: list[torch.Tensor], tokens: list[str]):
    """Analyze statistical properties of attention weights."""
    
    print("📊 Attention Statistics Analysis:")
    print("=" * 50)
    
    for layer_idx, layer_attn in enumerate(attention_weights):
        attn = layer_attn[0]  # Remove batch dimension
        num_heads = attn.shape[0]
        
        # Calculate statistics per layer
        mean_attn = attn.mean().item()
        std_attn = attn.std().item()
        max_attn = attn.max().item()
        min_attn = attn.min().item()
        
        print(f"\n🔍 Layer {layer_idx}:")
        print(f"  Heads: {num_heads}")
        print(f"  Mean attention: {mean_attn:.4f}")
        print(f"  Std attention: {std_attn:.4f}")
        print(f"  Max attention: {max_attn:.4f}")
        print(f"  Min attention: {min_attn:.4f}")
        
        # Find most attended tokens per head
        for head_idx in range(min(3, num_heads)):  # Show first 3 heads
            head_attn = attn[head_idx]
            max_indices = head_attn.sum(dim=0).argmax()
            most_attended_token = tokens[max_indices]
            print(f"  Head {head_idx} most attended: '{most_attended_token}'")

# Visualize attention patterns
print("🎨 Creating attention visualizations...")

# Plot specific attention head
plot_attention_head(attentions, tokens, layer_idx=0, head_idx=0)

# Plot overview
plot_attention_overview(attentions, tokens, max_layers=3, max_heads=4)

# Analyze statistics
analyze_attention_statistics(attentions, tokens)

## Alternative Scoring Functions

While scaled dot-product is the standard, other attention mechanisms exist:

1. **Additive Attention** (Bahdanau et al.): `score = v^T tanh(W_q Q + W_k K)`
2. **Bilinear Attention**: `score = Q W K^T`
3. **Sparse Attention**: Limited attention patterns (local, strided)
4. **Learned Position Bias**: Adding trainable position-dependent terms

Let's implement and compare these alternatives.

In [ ]:
class AdditiveAttention(nn.Module):
    """Bahdanau-style additive attention: score = v^T tanh(W_q Q + W_k K).

    The scores come from a small feed-forward net rather than a dot product, so
    queries and keys are free to live in different dimensions. The cost is that
    the [L_q, L_k, hidden] intermediate has to be materialised, which is why this
    fell out of favour once sequences got long.
    """

    def __init__(self, d_model: int, hidden_dim: int | None = None):
        super().__init__()
        hidden_dim = hidden_dim or d_model
        self.W_q = nn.Linear(d_model, hidden_dim, bias=False)
        self.W_k = nn.Linear(d_model, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, Q, K, V, mask=None):
        # [B, Lq, 1, H] + [B, 1, Lk, H] broadcast to [B, Lq, Lk, H]
        energy = torch.tanh(self.W_q(Q).unsqueeze(2) + self.W_k(K).unsqueeze(1))
        scores = self.v(energy).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        return weights @ V, weights


class BilinearAttention(nn.Module):
    """score = Q W K^T -- a learned matrix where dot-product attention uses identity.

    Strictly more expressive than the dot product (set W = I to recover it) at the
    cost of d_model^2 extra parameters per attention block.
    """

    def __init__(self, d_model: int):
        super().__init__()
        self.W = nn.Linear(d_model, d_model, bias=False)

    def forward(self, Q, K, V, mask=None):
        scores = self.W(Q) @ K.transpose(-2, -1) / math.sqrt(Q.size(-1))
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        return weights @ V, weights


class SparseLocalAttention(nn.Module):
    """Scaled dot-product restricted to a band of +/- window_size around the diagonal.

    This is the simplest member of the sparse-attention family (Longformer's sliding
    window is the same idea with global tokens added). The compute is still O(n^2)
    here because we build the full matrix and then mask it -- a production kernel
    would never materialise the masked entries at all. What the toy version does show
    honestly is the *sparsity* of the resulting weight matrix.
    """

    def __init__(self, window_size: int = 2):
        super().__init__()
        self.window_size = window_size

    def forward(self, Q, K, V, mask=None):
        d_k = Q.size(-1)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

        seq_len = Q.size(-2)
        positions = torch.arange(seq_len, device=Q.device)
        in_band = (positions.unsqueeze(1) - positions.unsqueeze(0)).abs() <= self.window_size
        scores = scores.masked_fill(~in_band, float('-inf'))

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        return weights @ V, weights


# Sanity check: every mechanism must return a [B, L, D] output and a row-stochastic
# [B, L, L] weight matrix, or the comparison below is not comparing like with like.
_Q = _K = _V = torch.randn(1, 8, 64)
for _name, _mech in [
    ('Additive', AdditiveAttention(64)),
    ('Bilinear', BilinearAttention(64)),
    ('Sparse Local', SparseLocalAttention(window_size=2)),
]:
    _out, _w = _mech(_Q, _K, _V)
    _row_sums = _w.sum(dim=-1)
    print(f'{_name:<14} output {tuple(_out.shape)}  weights {tuple(_w.shape)}  '
          f'row sums {_row_sums.min():.4f}-{_row_sums.max():.4f}')
print('\nAll three share the (Q, K, V) -> (output, weights) interface used below.')

In [ ]:
def compare_attention_mechanisms():
    """Compare different attention mechanisms."""
    print("🔄 Comparing Attention Mechanisms:")
    print("=" * 40)
    
    # Setup
    batch_size, seq_len, d_model = 1, 8, 64
    Q = torch.randn(batch_size, seq_len, d_model)
    K = torch.randn(batch_size, seq_len, d_model)
    V = torch.randn(batch_size, seq_len, d_model)
    
    # Define mechanisms with proper calling patterns
    mechanisms = {
        'Scaled Dot-Product': {
            'type': 'function',
            'implementation': lambda: scaled_dot_product_attention(Q, K, V)
        },
        'Additive': {
            'type': 'module',
            'implementation': AdditiveAttention(d_model)
        },
        'Bilinear': {
            'type': 'module', 
            'implementation': BilinearAttention(d_model)
        },
        'Sparse Local': {
            'type': 'module',
            'implementation': SparseLocalAttention(window_size=2)
        }
    }
    
    results = {}
    
    for name, config in mechanisms.items():
        start_time = time.time()
        
        try:
            if config['type'] == 'function':
                output, weights = config['implementation']()
            else:  # module
                output, weights = config['implementation'](Q, K, V)
            
            end_time = time.time()
            
            results[name] = {
                'output_shape': output.shape,
                'weights_shape': weights.shape,
                'time_ms': (end_time - start_time) * 1000,
                'weights_mean': weights.mean().item(),
                'weights_std': weights.std().item(),
                'sparsity': (weights < 0.01).float().mean().item()
            }
            
            print(f"\n📊 {name}:")
            print(f"  Output shape: {output.shape}")
            print(f"  Time: {results[name]['time_ms']:.2f}ms")
            print(f"  Weights mean: {results[name]['weights_mean']:.4f}")
            print(f"  Weights std: {results[name]['weights_std']:.4f}")
            print(f"  Sparsity (< 0.01): {results[name]['sparsity']:.2%}")
            
        except Exception as e:
            print(f"\n❌ {name} failed: {e}")
            results[name] = {'error': str(e)}
    
    return results

# Updated visualization code
def visualize_attention_mechanisms():
    """Visualize different attention mechanisms side by side."""
    print("🎨 Visualizing Attention Mechanisms...")
    
    batch_size, seq_len, d_model = 1, 8, 64
    Q = torch.randn(batch_size, seq_len, d_model)
    K = torch.randn(batch_size, seq_len, d_model) 
    V = torch.randn(batch_size, seq_len, d_model)
    
    mechanisms = [
        ('Scaled Dot-Product', lambda: scaled_dot_product_attention(Q, K, V)),
        ('Additive', AdditiveAttention(d_model)),
        ('Bilinear', BilinearAttention(d_model)),
        ('Sparse Local', SparseLocalAttention(window_size=2))
    ]
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    for idx, (name, mechanism) in enumerate(mechanisms):
        try:
            if callable(mechanism) and not isinstance(mechanism, nn.Module):
                # It's a lambda function
                _, weights = mechanism()
            else:
                # It's a PyTorch module
                _, weights = mechanism(Q, K, V)
            
            # Plot attention pattern
            im = axes[idx].imshow(weights[0].detach().numpy(), cmap='Blues', aspect='auto')
            axes[idx].set_title(f'{name} Attention', fontweight='bold')
            axes[idx].set_xlabel('Key Position')
            axes[idx].set_ylabel('Query Position')
            
            # Add colorbar
            plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)
            
        except Exception as e:
            axes[idx].text(0.5, 0.5, f'Error: {str(e)[:50]}...', 
                          ha='center', va='center', transform=axes[idx].transAxes,
                          bbox=dict(boxstyle="round,pad=0.3", facecolor="lightcoral"))
            axes[idx].set_title(f'{name} - Error', fontweight='bold', color='red')
    
    plt.tight_layout()
    plt.suptitle('Attention Pattern Comparison', fontsize=16, y=1.02, fontweight='bold')
    plt.show()

# Run the corrected comparison
comparison_results = compare_attention_mechanisms()

# Create visualizations
visualize_attention_mechanisms()

## Benchmarking Performance

We'll compare the computational efficiency of:
1. Our scratch implementations vs. optimized Hugging Face implementations
2. Single-head vs. multi-head attention
3. Different sequence lengths and model dimensions
4. Memory usage patterns

This helps understand the trade-offs between flexibility and performance.

### Evaluation protocol and metrics

Read the numbers below with the protocol in mind, because a benchmark is only as
trustworthy as its method:

- **Metric**: mean wall-clock milliseconds over 10 forward passes, plus an analytic memory
  estimate for the attention matrix (`batch x heads x L x L x 4` bytes for float32).
- **No warm-up pass.** The first iteration of each loop pays for lazy CUDA/BLAS init and
  memory allocation, and it is included in the mean. Expect the smallest configuration to
  look worse than it is.
- **No synchronisation.** On CPU this is fine; if you re-run on GPU you must call
  `torch.cuda.synchronize()` before reading the clock or you will be timing kernel launches.
- **Single sample per configuration.** There is no repetition or confidence interval, so
  treat differences under roughly 20% as noise, and only the order-of-magnitude gaps as real.

The comparison that matters is not scratch-versus-PyTorch in absolute terms -- it is the
*shape* of the curve as sequence length grows, which is where the quadratic cost shows up.

In [ ]:
def benchmark_attention_implementations():
    """Benchmark different attention implementations."""
    print("⚡ Benchmarking Attention Implementations")
    print("=" * 50)
    
    # Test configurations
    configs = [
        {'seq_len': 64, 'd_model': 256, 'num_heads': 8, 'batch_size': 16},
        {'seq_len': 128, 'd_model': 512, 'num_heads': 8, 'batch_size': 8},
        {'seq_len': 256, 'd_model': 768, 'num_heads': 12, 'batch_size': 4},
    ]
    
    results = []
    
    for config in configs:
        print(f"\n🔧 Configuration: {config}")
        
        seq_len = config['seq_len']
        d_model = config['d_model']
        num_heads = config['num_heads']
        batch_size = config['batch_size']
        
        # Create test data
        x = torch.randn(batch_size, seq_len, d_model)
        Q = torch.randn(batch_size, seq_len, d_model)
        K = torch.randn(batch_size, seq_len, d_model)
        V = torch.randn(batch_size, seq_len, d_model)
        
        config_results = {'config': config}
        
        # 1. Scratch single-head attention
        start_time = time.time()
        for _ in range(10):
            output, _ = scaled_dot_product_attention(Q, K, V)
        single_head_time = (time.time() - start_time) / 10
        config_results['single_head_ms'] = single_head_time * 1000
        
        # 2. Scratch multi-head attention
        start_time = time.time()
        for _ in range(10):
            output, _ = multi_head_attention(x, num_heads, d_model)
        multi_head_time = (time.time() - start_time) / 10
        config_results['multi_head_ms'] = multi_head_time * 1000
        
        # 3. PyTorch native multi-head attention
        mha = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        mha.eval()
        
        start_time = time.time()
        with torch.no_grad():
            for _ in range(10):
                output, _ = mha(x, x, x)
        pytorch_time = (time.time() - start_time) / 10
        config_results['pytorch_ms'] = pytorch_time * 1000
        
        # Memory usage (approximate)
        attention_memory = batch_size * num_heads * seq_len * seq_len * 4  # bytes
        config_results['memory_mb'] = attention_memory / (1024 * 1024)
        
        # Speedup calculations
        config_results['pytorch_speedup'] = single_head_time / pytorch_time
        
        results.append(config_results)
        
        print(f"  Single-head: {config_results['single_head_ms']:.2f}ms")
        print(f"  Multi-head:  {config_results['multi_head_ms']:.2f}ms")
        print(f"  PyTorch MHA: {config_results['pytorch_ms']:.2f}ms")
        print(f"  Memory:      {config_results['memory_mb']:.1f}MB")
        print(f"  Speedup:     {config_results['pytorch_speedup']:.1f}x")
    
    return results

def plot_benchmark_results(results: list[dict]):
    """Visualize benchmark results."""
    
    configs = [f"L{r['config']['seq_len']}_D{r['config']['d_model']}" for r in results]
    single_head_times = [r['single_head_ms'] for r in results]
    multi_head_times = [r['multi_head_ms'] for r in results]
    pytorch_times = [r['pytorch_ms'] for r in results]
    memory_usage = [r['memory_mb'] for r in results]
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
    
    # Timing comparison
    x = np.arange(len(configs))
    width = 0.25
    
    ax1.bar(x - width, single_head_times, width, label='Single-head', alpha=0.8)
    ax1.bar(x, multi_head_times, width, label='Multi-head', alpha=0.8)
    ax1.bar(x + width, pytorch_times, width, label='PyTorch MHA', alpha=0.8)
    
    ax1.set_xlabel('Configuration')
    ax1.set_ylabel('Time (ms)')
    ax1.set_title('Attention Implementation Timing')
    ax1.set_xticks(x)
    ax1.set_xticklabels(configs, rotation=45)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Speedup comparison
    speedups = [r['pytorch_speedup'] for r in results]
    ax2.bar(configs, speedups, color='orange', alpha=0.7)
    ax2.set_xlabel('Configuration')
    ax2.set_ylabel('Speedup Factor')
    ax2.set_title('PyTorch vs. Scratch Implementation')
    ax2.grid(axis='y', alpha=0.3)
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)
    
    # Memory usage
    ax3.plot(configs, memory_usage, marker='o', linewidth=2, markersize=8, color='red')
    ax3.set_xlabel('Configuration')
    ax3.set_ylabel('Memory (MB)')
    ax3.set_title('Attention Memory Usage')
    ax3.grid(alpha=0.3)
    plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45)
    
    # Complexity analysis
    seq_lens = [r['config']['seq_len'] for r in results]
    theoretical_complexity = [(s**2) / 1000 for s in seq_lens]  # O(n²) normalized
    
    ax4.plot(seq_lens, pytorch_times, 'o-', label='Actual Time', linewidth=2)
    ax4.plot(seq_lens, theoretical_complexity, '--', label='O(n²) Theoretical', linewidth=2)
    ax4.set_xlabel('Sequence Length')
    ax4.set_ylabel('Time (ms) / Complexity')
    ax4.set_title('Attention Complexity')
    ax4.legend()
    ax4.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Run benchmarks
benchmark_results = benchmark_attention_implementations()
plot_benchmark_results(benchmark_results)

## Case Study: Translation or Classification

Let's apply attention inspection to a real NLP task to see how attention patterns relate to model performance and linguistic understanding.

We'll:
1. Load a pre-trained model for a specific task
2. Analyze attention patterns on meaningful examples
3. Correlate attention weights with task-relevant features
4. Identify which heads capture different linguistic phenomena

In [ ]:
def analyze_sentiment_attention():
    """Analyze attention patterns in sentiment classification."""
    from transformers import AutoModelForSequenceClassification
    
    print("🎭 Analyzing Sentiment Classification Attention")
    print("=" * 50)
    
    # Load sentiment model
    model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, output_attentions=True)
    model.eval()
    
    # Test sentences with clear sentiment indicators
    test_sentences = [
        "I absolutely love this amazing product!",
        "This is terrible and completely disappointing.",
        "The movie was okay, nothing special but not bad either.",
        "Fantastic service, highly recommend to everyone!"
    ]
    
    results = []
    
    for sentence in test_sentences:
        print(f"\n🔍 Analyzing: '{sentence}'")
        
        # Tokenize and predict
        inputs = tokenizer(sentence, return_tensors='pt', padding=True)
        tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        predictions = torch.softmax(outputs.logits, dim=-1)
        predicted_class = predictions.argmax().item()
        confidence = predictions.max().item()
        
        # Labels: 0=negative, 1=neutral, 2=positive
        sentiment_labels = ['Negative', 'Neutral', 'Positive']
        
        print(f"  Prediction: {sentiment_labels[predicted_class]} ({confidence:.3f})")
        print(f"  Tokens: {tokens}")
        
        # Analyze attention patterns
        attentions = outputs.attentions
        
        # Find tokens with highest attention (averaged across heads/layers)
        avg_attention = torch.stack(attentions).mean(dim=(0, 2))  # Average over layers and heads
        total_attention = avg_attention.sum(dim=0)  # Sum over query positions
        
        # Get top attended tokens
        top_indices = total_attention.argsort(descending=True)[:3].tolist()  # .tolist():
        # argsort returns 0-dim tensors, and `tokens` is a Python list, so
        # `tokens[i]` raised TypeError and this whole section never ran on Kaggle.
        top_tokens = [tokens[i] for i in top_indices]
        top_weights = [total_attention[i].item() for i in top_indices]
        
        print(f"  Top attended tokens: {list(zip(top_tokens, [f'{w:.3f}' for w in top_weights]))}")
        
        results.append({
            'sentence': sentence,
            'prediction': sentiment_labels[predicted_class],
            'confidence': confidence,
            'tokens': tokens,
            'top_tokens': top_tokens,
            'attentions': attentions
        })
    
    return results

def visualize_task_attention(results: list):
    """Visualize attention patterns for the task."""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    for idx, result in enumerate(results[:4]):
        tokens = result['tokens']
        attentions = result['attentions']
        
        # Average attention across layers and heads
        avg_attn = torch.stack(attentions).mean(dim=(0, 2))[0].numpy()
        
        # Plot heatmap
        im = axes[idx].imshow(avg_attn, cmap='Reds', aspect='auto')
        axes[idx].set_title(f"{result['prediction']}: {result['sentence'][:30]}...", 
                           fontsize=10, fontweight='bold')
        
        # Set ticks
        axes[idx].set_xticks(range(len(tokens)))
        axes[idx].set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
        axes[idx].set_yticks(range(len(tokens)))
        axes[idx].set_yticklabels(tokens, fontsize=8)
        
        # Add colorbar
        plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)
    
    plt.suptitle('Attention Patterns in Sentiment Classification', fontsize=16)
    plt.tight_layout()
    plt.show()

# Run sentiment analysis
sentiment_results = analyze_sentiment_attention()
visualize_task_attention(sentiment_results)

## Attention Head Ablation Study

Head ablation reveals which attention heads are most important for task performance. By systematically disabling individual heads and measuring performance drops, we can:

- **Identify critical heads** for specific tasks
- **Understand head specialization** patterns  
- **Optimize model efficiency** by pruning less important heads
- **Analyze redundancy** across attention heads

In [ ]:
def attention_head_ablation_study():
    """Ablate individual attention heads and measure how far the output moves.

    Hugging Face models accept a `head_mask` of shape [num_layers, num_heads]; a
    zero entry multiplies that head's context vector out of the sum before the
    output projection. That is a genuine ablation, and it is why this function
    uses it instead of a forward hook -- a hook on the attention module sees a
    plain tuple, so mutating an `output.attentions` attribute silently does
    nothing and every head scores as unimportant.
    """
    print("Attention Head Ablation Study")
    print("=" * 40)

    model_name = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()

    test_texts = [
        "The capital of France is Paris.",
        "Machine learning helps solve complex problems.",
        "Natural language processing is fascinating.",
        "Deep learning models require large datasets.",
    ]

    def sentence_embeddings(head_mask=None):
        """Mean-pooled last hidden state for each test sentence."""
        embeddings = []
        for text in test_texts:
            inputs = tokenizer(text, return_tensors='pt')
            with torch.no_grad():
                outputs = model(**inputs, head_mask=head_mask)
            embeddings.append(outputs.last_hidden_state.mean(dim=1))
        return torch.cat(embeddings, dim=0)

    baseline_embeddings = sentence_embeddings()

    num_layers = len(model.transformer.layer)
    num_heads = model.config.num_attention_heads
    print(f"Model has {num_layers} layers x {num_heads} heads = "
          f"{num_layers * num_heads} heads in total")

    layers_to_test = min(3, num_layers)
    heads_to_test = min(4, num_heads)
    print(f"Ablating the first {layers_to_test} layers x {heads_to_test} heads "
          f"= {layers_to_test * heads_to_test} ablations\n")

    ablation_results = []
    for layer_idx in range(layers_to_test):
        for head_idx in range(heads_to_test):
            head_mask = torch.ones(num_layers, num_heads)
            head_mask[layer_idx, head_idx] = 0.0

            ablated_embeddings = sentence_embeddings(head_mask)

            similarity = F.cosine_similarity(
                baseline_embeddings, ablated_embeddings
            ).mean().item()
            drop = 1 - similarity

            ablation_results.append({
                'layer': layer_idx,
                'head': head_idx,
                'similarity': similarity,
                'performance_drop': drop,
            })
            print(f"  L{layer_idx}H{head_idx}: similarity={similarity:.4f}, drop={drop:.4f}")

    # A control: if the masking works, zeroing an entire layer must move the
    # embedding much further than zeroing any single head in it.
    whole_layer_mask = torch.ones(num_layers, num_heads)
    whole_layer_mask[0] = 0.0
    layer_drop = 1 - F.cosine_similarity(
        baseline_embeddings, sentence_embeddings(whole_layer_mask)
    ).mean().item()
    biggest_head_drop = max(r['performance_drop'] for r in ablation_results)
    print(f"\nControl -- whole of layer 0 ablated: drop={layer_drop:.4f}")
    print(f"Largest single-head drop:            {biggest_head_drop:.4f}")
    print("The layer drop should dominate; if both are ~0 the mask is not being applied.")

    return ablation_results


def visualize_ablation_results(ablation_results: list):
    """Visualize head importance from ablation study."""
    
    # Convert to arrays for plotting
    layers = [r['layer'] for r in ablation_results]
    heads = [r['head'] for r in ablation_results]
    drops = [r['performance_drop'] for r in ablation_results]
    
    # Create heatmap
    max_layer = max(layers) + 1
    max_head = max(heads) + 1
    
    heatmap_data = np.zeros((max_layer, max_head))
    for layer, head, drop in zip(layers, heads, drops):
        heatmap_data[layer, head] = drop
    
    plt.figure(figsize=(10, 6))
    sns.heatmap(heatmap_data, 
                annot=True, 
                fmt='.3f',
                cmap='Reds',
                xticklabels=[f'Head {i}' for i in range(max_head)],
                yticklabels=[f'Layer {i}' for i in range(max_layer)],
                cbar_kws={'label': 'Performance Drop'})
    
    plt.title('Attention Head Importance (Ablation Study)', fontsize=14, fontweight='bold')
    plt.xlabel('Attention Head')
    plt.ylabel('Layer')
    plt.tight_layout()
    plt.show()
    
    # Show most/least important heads
    sorted_results = sorted(ablation_results, key=lambda x: x['performance_drop'], reverse=True)
    
    print("\n🏆 Most Important Heads (highest performance drop):")
    for i, result in enumerate(sorted_results[:5]):
        print(f"  {i+1}. Layer {result['layer']}, Head {result['head']}: {result['performance_drop']:.4f}")
    
    print("\n🔧 Least Important Heads (lowest performance drop):")
    for i, result in enumerate(sorted_results[-5:]):
        print(f"  {i+1}. Layer {result['layer']}, Head {result['head']}: {result['performance_drop']:.4f}")

# Run ablation study
ablation_results = attention_head_ablation_study()
visualize_ablation_results(ablation_results)

## Limitations & Caveats

Everything above is a forward pass and a plot, and forward passes are easy to over-read.
Six things to hold on to before you quote any of it.

**1. An attention map is not an explanation.** This is the big one. High attention weight on
a token means that token's value vector contributed to the output at that position; it does
not mean the model's decision depended on it. Jain and Wallace's *Attention is not
Explanation* showed you can often find a very different attention distribution that produces
an identical prediction, and Wiegreffe and Pinter's *Attention is not not Explanation*
narrowed but did not remove the objection. The sentiment case study above is therefore a
demonstration of where attention mass lands, not evidence of why the model predicted what it
did. For a causal claim you need an intervention -- which is what the ablation section is
for.

**2. Averaging across heads and layers destroys the thing you are looking at.** The case
study collapses 12 layers x 12 heads into one matrix before plotting. Heads specialise, and
several well-documented ones (previous-token, delimiter-attending) point in near-opposite
directions, so the mean of all of them is a blur that no individual head resembles. Plot
single heads when you want to see structure; the averaged view is only useful for spotting
which *tokens* soak up attention overall.

**3. `multi_head_attention` re-initialises its projections on every call.** Look at the
function: `W_q`, `W_k`, `W_v` and `W_o` are constructed inside `forward`, so each invocation
uses fresh random weights and no gradient ever reaches them. That is fine for what the cell
uses it for -- checking shapes and measuring the cost of the reshape-and-project dance -- but
its *outputs* are noise and its benchmark numbers include the cost of building four linear
layers, which a real model pays once at init. Do not read it as a working attention block.

**4. The benchmark measures Python, not attention.** The scratch implementation is slower
than `nn.MultiheadAttention` mostly because the latter fuses the QKV projections into one
matmul and dispatches to a hand-written kernel, not because the maths is different. On CPU
with short sequences you are largely timing interpreter overhead, and with no warm-up and a
single trial per configuration the small differences are noise. The memory column is an
analytic estimate of the attention matrix alone -- it excludes activations, weights and
autograd state, so it is a lower bound, not a measurement.

**5. The ablation is a drift measurement on four sentences, not a task metric.** Cosine
distance in a mean-pooled embedding space is a proxy: it tells you a head changed the
representation, not that anything downstream got worse. A head could be critical for a task
this probe never touches, and four short factual sentences exercise a narrow slice of the
model. The standard version of this experiment measures accuracy drop on a real evaluation
set; treat the heatmap here as a screening tool that says where to look.

**6. Single-head ablation understates redundancy.** Heads are redundant, so removing one at a
time usually shows small drops even for heads that matter -- the others compensate. The
control in the ablation cell makes the point: zeroing a whole layer moves the embedding far
more than the sum of its individual heads would suggest. If you want to know how many heads a
model actually needs, prune greedily and cumulatively rather than one at a time.

**7. Nothing here generalises past these two checkpoints.** DistilBERT is a distilled,
6-layer encoder and the sentiment model is a RoBERTa fine-tune. Decoder-only models with
causal masks, models with rotary or ALiBi position encodings, and anything using grouped-query
or sliding-window attention will show visibly different patterns. Re-run the notebook against
the checkpoint you actually care about before drawing conclusions about it.

## Conclusion & Next Steps

### 🎯 Key Insights from Our Analysis

1. **Attention Mechanisms**: We implemented and compared multiple attention variants, showing trade-offs between complexity and performance

2. **Visualization Patterns**: Different attention heads capture distinct linguistic phenomena - some focus on syntax, others on semantics

3. **Performance Trade-offs**: PyTorch's optimized implementations are significantly faster than scratch versions, but custom implementations offer more flexibility

4. **Head Specialization**: Ablation studies reveal that different heads have varying importance, suggesting opportunities for model compression

5. **Task Adaptation**: Attention mass in the sentiment model concentrates on emotion-bearing tokens. Read that as *where the weights went*, not as *why the model predicted what it did* -- see caveat 1 in the previous section

### 🚀 Further Explorations

- **Sparse Transformers**: Implement efficient sparse attention patterns (Longformer, BigBird)
- **Funnel Transformers**: Explore progressive dimension reduction
- **Cross-Attention**: Analyze encoder-decoder attention in translation models  
- **Attention Interventions**: Modify attention weights to test causal relationships
- **Multi-Scale Attention**: Combine local and global attention patterns

### 📚 Further Reading

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) - Original Transformer paper
- [Hugging Face Transformers Documentation](https://huggingface.co/docs/transformers/)
- [The Illustrated Transformer](http://jalammar.github.io/illustrated-transformer/)
- [Attention Head Analysis](https://arxiv.org/abs/1905.09418) - What do heads learn?

**Happy exploring! 🔍**